# One-Hot Encoding (OHE)

Welcome to the **One-Hot Encoding** practical study notebook.

Machine learning models are mathematical equations that calculate weighted sums and geometric distances. They cannot multiply a weight by `"Red"`, `"Delhi"`, or `"Male"`.

In this notebook, we follow our core learning pattern:
```text
Concept
   ↓
Small manual example
   ↓
Code implementation
   ↓
Output
   ↓
Interpretation
```

### What We Will Cover:
1. **Understanding categorical data** and why naive integer encoding breaks models.
2. **Manual One-Hot Encoding** (how the logic works underneath).
3. **One-Hot Encoding with Pandas** (`pd.get_dummies`).
4. **The Dummy Variable Trap & `drop_first=True`**.
5. **Production One-Hot Encoding with Scikit-Learn** (`OneHotEncoder`).
6. **Understanding `fit()`, `transform()`, and preventing Data Leakage**.
7. **Handling Unseen Categories** with `handle_unknown="ignore"`.
8. **High Cardinality & Dimensionality Explosion**.
9. **Rare Category Grouping** (`"Other"` category strategy).
10. **OHE vs. Ordinal Encoding comparison & Decision Flow**.

---


## 1. Understanding Categorical Data

Let us start with a simple toy dataset of car colors to observe raw categorical labels.


In [1]:
import pandas as pd
import numpy as np

# WHY THIS IS REQUIRED:
# Create a simple toy dataset to see how categorical text columns appear in real data.

car_data = pd.DataFrame({
    'Car_ID': [101, 102, 103, 104],
    'Color': ['Red', 'Blue', 'Green', 'Red']
})

car_data


,Car_ID,Color
0,101,Red
1,102,Blue
2,103,Green
3,104,Red


### Observation
The `Color` column contains text strings.
If we attempt to train a mathematical model directly on this column, it fails because linear equations cannot compute $w_1 \times \text{"Red"}$.


## 2. Why Ordinary Integer Encoding Can Be Problematic

A natural first thought is: *"Why not just replace each color with a number?"*
- `Red` $\rightarrow$ 0
- `Blue` $\rightarrow$ 1
- `Green` $\rightarrow$ 2

Let's test this in Python and calculate the mathematical distances between these encoded colors.


In [2]:
# WHY THIS IS REQUIRED:
# Demonstrate how naive integer encoding introduces artificial distances and false rankings.
#
# IN BASIC TERMS:
# We show that the math assumes Green (2) is twice as far from Red (0) as Blue (1) is, which is false!

integer_encoded = car_data.copy()
integer_encoded['Color_Integer'] = [0, 1, 2, 0]

# Calculate mathematical distances
dist_green_red = abs(2 - 0)
dist_blue_red = abs(1 - 0)

print(integer_encoded[['Car_ID', 'Color', 'Color_Integer']])
print(f"\nMath says: Distance between Green and Red = {dist_green_red}")
print(f"Math says: Distance between Blue and Red  = {dist_blue_red}")


   Car_ID  Color  Color_Integer
0     101    Red              0
1     102   Blue              1
2     103  Green              2
3     104    Red              0

Math says: Distance between Green and Red = 2
Math says: Distance between Blue and Red  = 1


### Interpretation: The Serious Flaw
The math assumes:
1. **Green is greater than Blue, which is greater than Red** ($2 > 1 > 0$).
2. **Green is twice as far from Red as Blue is** ($2 - 0 = 2$ vs $1 - 0 = 1$).

Colors have **no natural hierarchy or order**.
This naive mapping introduces a completely fake mathematical relationship that corrupts distance-based and linear models.


## 3. Manual One-Hot Encoding

Instead of one column with integers, One-Hot Encoding creates a **separate binary (0/1) column for every unique category**:
- `Red` $\rightarrow [1,\; 0,\; 0]$
- `Blue` $\rightarrow [0,\; 1,\; 0]$
- `Green` $\rightarrow [0,\; 0,\; 1]$

Let's write a pure Python implementation to understand the mechanism before using libraries.


In [3]:
# WHY THIS IS REQUIRED:
# Implement One-Hot Encoding manually to see the underlying logic step-by-step.
#
# IN BASIC TERMS:
# Find all unique categories, create a column for each, and mark 1 if the row matches.

colors_list = ['Red', 'Blue', 'Green', 'Red']

# Step 1: Find unique categories
unique_categories = sorted(list(set(colors_list)))
print(f"Unique categories found: {unique_categories}")

# Step 2: Build binary matrix manually
manual_ohe = {}
for cat in unique_categories:
    # 1 if item matches category, 0 otherwise
    manual_ohe[f"Color_{cat}"] = [1 if item == cat else 0 for item in colors_list]

manual_ohe_df = pd.DataFrame(manual_ohe)
pd.concat([pd.Series(colors_list, name='Original_Color'), manual_ohe_df], axis=1)


Unique categories found: ['Blue', 'Green', 'Red']


,Original_Color,Color_Blue,Color_Green,Color_Red
0,Red,0,0,1
1,Blue,1,0,0
2,Green,0,1,0
3,Red,0,0,1


### Interpretation
Every category now has its own binary column:
- Exactly **one column is 1 ("hot")** for each row, while all others are **0 ("cold")**.
- Now compute Euclidean distance between Red and Blue: $\sqrt{(1-0)^2 + (0-1)^2 + (0-0)^2} = \sqrt{2} \approx 1.414$.
- Between Red and Green: $\sqrt{(1-0)^2 + (0-0)^2 + (0-1)^2} = \sqrt{2} \approx 1.414$.
**Every color is now equidistant from every other color!** Zero artificial ordering exists.


## 4. One-Hot Encoding Using Pandas: `pd.get_dummies()`

Pandas has a built-in function called `pd.get_dummies()` designed for quick exploratory data analysis.


In [4]:
# WHY THIS IS REQUIRED:
# Learn the Pandas way of creating dummy variables for rapid data exploration.

df_colors = pd.DataFrame({
    'Color': ['Red', 'Blue', 'Green', 'Red']
})

# Apply pd.get_dummies (dtype=int ensures clean 0/1 integers instead of True/False)
df_dummies = pd.get_dummies(df_colors, columns=['Color'], dtype=int)

df_dummies


,Color_Blue,Color_Green,Color_Red
0,0,0,1
1,1,0,0
2,0,1,0
3,0,0,1


### Interpretation
Pandas automatically identified the 3 unique colors and created 3 binary columns: `Color_Blue`, `Color_Green`, and `Color_Red`.


## 5. The Dummy Variable Trap & `drop_first=True`

### Understand It First
Consider a feature named `Gender` with two categories: `Male` and `Female`.
If we create two columns (`Gender_Male` and `Gender_Female`):
- When `Gender_Male` is 1, `Gender_Female` is guaranteed to be 0.
- When `Gender_Male` is 0, `Gender_Female` is guaranteed to be 1.

$$\text{Gender\_Female} = 1 - \text{Gender\_Male}$$

The two columns are **100% linearly dependent** (perfect multicollinearity, $r = -1.0$).
In linear regression or logistic regression, this creates a mathematical failure known as the **Dummy Variable Trap**.

### The Solution: $N - 1$ Columns
If a categorical variable has $N$ categories, we only need **$N - 1$ columns**.
The dropped category becomes the **baseline/reference category**.


In [5]:
# WHY THIS IS REQUIRED:
# Demonstrate drop_first=True to avoid the Dummy Variable Trap.
#
# IN BASIC TERMS:
# We drop 1 column. If all remaining columns are 0, it means the row belongs to the dropped category!

df_drop_first = pd.get_dummies(df_colors, columns=['Color'], drop_first=True, dtype=int)

pd.concat([df_colors, df_drop_first], axis=1)


,Color,Color_Green,Color_Red
0,Red,0,1
1,Blue,0,0
2,Green,1,0
3,Red,0,1


### Interpretation
Notice:
- `Color_Blue` was dropped as the reference baseline.
- When `Color_Green == 0` and `Color_Red == 0`, both columns are 0 $\longrightarrow$ **The color must be Blue!**
- We saved 1 column without losing any information.


## 6. One-Hot Encoding with Scikit-Learn: `OneHotEncoder`

While `pd.get_dummies()` is convenient for quick analysis, **Scikit-Learn's `OneHotEncoder` is required for production machine learning pipelines**.

### Why Scikit-Learn's `OneHotEncoder` is Preferred for ML:
1. **Pipeline integration:** Fits on training data and transforms test data with identical column mappings.
2. **Prevents shape mismatches:** If test data is missing a category, `OneHotEncoder` maintains the correct training columns.
3. **Handles unseen categories gracefully:** Built-in support for `handle_unknown="ignore"`.


In [6]:
from sklearn.preprocessing import OneHotEncoder

# WHY THIS IS REQUIRED:
# Learn the standard Scikit-Learn implementation used in machine learning pipelines.
#
# IN BASIC TERMS:
# We use sparse_output=False to get a standard NumPy array, and drop='first' to avoid multicollinearity.

ohe_sk = OneHotEncoder(sparse_output=False, drop='first')

# Fit and transform (requires 2D input like a DataFrame)
encoded_array = ohe_sk.fit_transform(df_colors[['Color']])

# Extract generated feature names
feature_names = ohe_sk.get_feature_names_out(['Color'])

# Construct a readable DataFrame
pd.DataFrame(encoded_array, columns=feature_names)


,Color_Green,Color_Red
0,0.0,1.0
1,0.0,0.0
2,1.0,0.0
3,0.0,1.0


### Interpretation
`OneHotEncoder` dropped `'Blue'` (first alphabetically) and outputted 2 columns: `Color_Green` and `Color_Red`.


## 7. Understanding `fit()`, `transform()`, and `fit_transform()`

In Scikit-Learn:
- **`fit(X_train)`**: Scans the data to **learn** the unique categories present in the training set.
- **`transform(X)`**: Uses the previously learned categories to **convert** rows into binary columns.
- **`fit_transform(X_train)`**: Performs both in a single step on training data.

```text
Training Data
      ↓
    fit()
      ↓
Learn categories
      ↓
  transform()
      ↓
Encoded Training Data

Test Data (Unseen)
      ↓
transform() using SAME learned categories
      ↓
Encoded Test Data
```


## 8. Train/Test Workflow & Preventing Data Leakage

### What is Data Leakage in Preprocessing?
If you fit your encoder on the entire dataset *before* splitting:
- Categories and distributions from the test set influence the encoding.
- Your test set is no longer truly unseen.

### The Correct Pipeline:
1. Split raw data first: `X_train`, `X_test`.
2. Call `fit_transform(X_train)`.
3. Call `transform(X_test)`.


In [7]:
from sklearn.model_selection import train_test_split

# WHY THIS IS REQUIRED:
# Demonstrate the correct train/test split workflow that guarantees no data leakage.

dataset = pd.DataFrame({
    'City': ['Delhi', 'Mumbai', 'Chennai', 'Delhi', 'Mumbai', 'Chennai', 'Delhi', 'Mumbai'],
    'Experience': [1, 3, 2, 5, 4, 6, 2, 5],
    'Hired': [0, 1, 0, 1, 1, 1, 0, 1]
})

X = dataset[['City', 'Experience']]
y = dataset['Hired']

# Step 1: Split raw data first
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Step 2: Initialize encoder and fit on X_train ONLY
encoder_pipeline = OneHotEncoder(sparse_output=False, drop='first')
X_train_city = encoder_pipeline.fit_transform(X_train[['City']])

# Step 3: Transform X_test using the SAME fitted encoder
X_test_city = encoder_pipeline.transform(X_test[['City']])

print(f"Categories learned from training: {encoder_pipeline.categories_[0].tolist()}")
print(f"X_train encoded shape: {X_train_city.shape}")
print(f"X_test encoded shape:  {X_test_city.shape}")


Categories learned from training: ['Chennai', 'Delhi', 'Mumbai']
X_train encoded shape: (6, 2)
X_test encoded shape:  (2, 2)


### Interpretation
The encoder learned the categories from `X_train` alone.
Both `X_train` and `X_test` received identical column structures without any test set information leaking during preprocessing.


## 9. Handling Unseen Categories with `handle_unknown="ignore"`

In real-world deployment, what happens if an unseen category appears in the test data?
- **Training set:** Contains only `Delhi` and `Mumbai`.
- **New incoming customer:** Located in `Bangalore`!

By default, Scikit-Learn will raise an error and crash your application.
Setting `handle_unknown="ignore"` instructs the encoder to safely encode unseen categories as **all zeros** (`[0, 0]`).


In [8]:
# WHY THIS IS REQUIRED:
# Show how handle_unknown="ignore" prevents runtime crashes when test data contains new categories.

train_cities = pd.DataFrame({'City': ['Delhi', 'Mumbai', 'Delhi']})
test_cities = pd.DataFrame({'City': ['Delhi', 'Mumbai', 'Bangalore']}) # 'Bangalore' was never seen!

# Initialize encoder with handle_unknown='ignore'
# (Note: drop='first' is mutually exclusive with handle_unknown='ignore' in sklearn to avoid ambiguity)
robust_ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit on training cities
robust_ohe.fit(train_cities[['City']])

# Transform test cities containing 'Bangalore'
encoded_test = robust_ohe.transform(test_cities[['City']])

# Display result
test_cols = robust_ohe.get_feature_names_out(['City'])
encoded_test_df = pd.DataFrame(encoded_test, columns=test_cols)
pd.concat([test_cities, encoded_test_df], axis=1)


,City,City_Delhi,City_Mumbai
0,Delhi,1.0,0.0
1,Mumbai,0.0,1.0
2,Bangalore,0.0,0.0


### Interpretation
Look at row index 2 for `'Bangalore'`:
- `City_Delhi` = **0.0**
- `City_Mumbai` = **0.0**

The encoder did not crash! It recognized that Bangalore was not in the training set and assigned 0 to all known city indicators.


## 10. High Cardinality & Dimensionality Explosion

### Understand It First
**Cardinality** means the count of unique categories in a feature.
If a column has 10,000 unique values (e.g. ZIP codes or product IDs), One-Hot Encoding creates **10,000 new columns**!

This leads to **Dimensionality Explosion**:
- Memory usage skyrockets.
- Training speed crawls.
- The model suffers from the "curse of dimensionality" and overfits.


In [9]:
# WHY THIS IS REQUIRED:
# Demonstrate how quickly high-cardinality data causes dimensionality explosion.

np.random.seed(42)
# Simulate 100 observations across 30 unique cities
sample_cities = [f"City_{i}" for i in np.random.randint(1, 31, size=100)]
high_card_df = pd.DataFrame({'City': sample_cities})

print(f"Number of rows: {len(high_card_df)}")
print(f"Number of unique categories: {high_card_df['City'].nunique()}")

# If we One-Hot Encode this directly:
ohe_high = OneHotEncoder(sparse_output=False)
transformed = ohe_high.fit_transform(high_card_df[['City']])

print(f"Dataset shape after One-Hot Encoding: {transformed.shape}")
print(f"We created {transformed.shape[1]} columns from a single feature!")


Number of rows: 100
Number of unique categories: 29
Dataset shape after One-Hot Encoding: (100, 29)
We created 29 columns from a single feature!


## 11. Rare Category Grouping (The `"Other"` Strategy)

To prevent dimensionality explosion, we group infrequent categories into a single label: `"Other"`.

### Important Rule:
There is **no universal magic threshold** (e.g., count < 100 or frequency < 5%).
The threshold depends on:
- Total dataset size
- Business context
- Distribution of category frequencies

Let us demonstrate how to group categories that appear infrequently into `"Other"`.


In [10]:
# WHY THIS IS REQUIRED:
# Demonstrate how to replace infrequent categories with 'Other' before encoding to keep dimensionality manageable.

city_counts = high_card_df['City'].value_counts()
print("Top 5 city counts:")
print(city_counts.head(5))
print("\nBottom 5 rare city counts:")
print(city_counts.tail(5))

# Rule: Keep cities appearing at least 5 times; group the rest into 'Other'
threshold = 5
frequent_cities = city_counts[city_counts >= threshold].index.tolist()

# Replace rare categories with 'Other'
high_card_df['City_Grouped'] = high_card_df['City'].apply(
    lambda c: c if c in frequent_cities else 'Other'
)

print(f"\nUnique categories before grouping: {high_card_df['City'].nunique()}")
print(f"Unique categories after grouping:  {high_card_df['City_Grouped'].nunique()}")

# Now One-Hot Encode the grouped feature
ohe_grouped = OneHotEncoder(sparse_output=False)
grouped_encoded = ohe_grouped.fit_transform(high_card_df[['City_Grouped']])
print(f"Columns created: {grouped_encoded.shape[1]} (Reduced from {transformed.shape[1]}!)")


Top 5 city counts:
City
City_28    6
City_12    6
City_21    6
City_7     5
City_26    5
Name: count, dtype: int64

Bottom 5 rare city counts:
City
City_9     2
City_14    2
City_1     1
City_16    1
City_5     1
Name: count, dtype: int64

Unique categories before grouping: 29
Unique categories after grouping:  11
Columns created: 11 (Reduced from 29!)


### Interpretation
By grouping rare categories, we reduced the column count from **30** down to **10**, slashing model complexity and memory usage while keeping the primary city signals intact!


## 12. OHE vs. Ordinal Encoding: Summary Comparison

| Feature Example | Data Type | Meaningful Order? | Recommended Encoder |
| :--- | :--- | :---: | :--- |
| **`Education`** (High School, UG, PG) | Ordinal | **Yes** | **`OrdinalEncoder`** |
| **`Rating`** (Poor, Average, Good) | Ordinal | **Yes** | **`OrdinalEncoder`** |
| **`City`** (Delhi, Mumbai, Chennai) | Nominal | **No** | **`OneHotEncoder`** |
| **`Color`** (Red, Blue, Green) | Nominal | **No** | **`OneHotEncoder`** |
| **`State`** (NY, CA, TX, FL) | Nominal | **No** | **`OneHotEncoder`** |
| **`Gender`** (Male, Female) | Nominal | **No** | **`OneHotEncoder(drop='first')`** |

> **The Simple Rule:**  
> - **Order exists?** $\longrightarrow$ **Ordinal Encoding**  
> - **No order?** $\longrightarrow$ **One-Hot Encoding**


# 13. Complete Categorical Encoding Decision Flow

```text
                           Categorical Feature
                                    │
                         Is it the Target (y)?
                                    │
                    ┌───────────────┴───────────────┐
                    ▼                               ▼
                   YES                              NO
                    │                               │
              LabelEncoder                Is there a meaningful
             (Pass=1, Fail=0)                     order?
                                                    │
                                    ┌───────────────┴───────────────┐
                                    ▼                               ▼
                                   YES                              NO
                                    │                               │
                              Ordinal Data                     Nominal Data
                                    │                               │
                              OrdinalEncoder                        │
                        (categories=[...])                          │
                                                           Is cardinality high?
                                                            (> 20 categories)
                                                                    │
                                                    ┌───────────────┴───────────────┐
                                                    ▼                               ▼
                                                   YES                              NO
                                                    │                               │
                                          Rare Grouping /                    OneHotEncoder
                                          Target Encoding                  (drop='first' or
                                                                        handle_unknown='ignore')
```

---

## What Did We Learn?

### 1. The Core Mental Model
```text
Original:
Color
-----
Red
Blue
Green

        ↓
One-Hot Encoding
        ↓

Color_Red   Color_Blue   Color_Green
    1           0             0
    0           1             0
    0           0             1
```

### 2. Core Takeaways to Remember:
1. **OHE converts nominal text into binary (0/1) indicator columns**, ensuring categories are equidistant without artificial ranking.
2. **Never use integer encoding on nominal data:** Doing so creates false linear relationships ($2 > 1 > 0$).
3. **The Dummy Variable Trap:** $N$ categories create $N - 1$ degrees of freedom. Drop one column (`drop='first'`) in linear models to prevent multicollinearity.
4. **Use Scikit-Learn in pipelines:** `OneHotEncoder` learns from training data and preserves exact column alignment on test data.
5. **Handle unseen categories:** Use `handle_unknown="ignore"` to encode new real-world categories as all zeros without runtime crashes.
6. **High Cardinality:** Group infrequent categories into `"Other"` to prevent dimensionality explosion.
